# Exercise 1.4.2.11 — (2/3) - implement `build_intermediate_nodes`

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.4.2 SAE Circuits`  
**Notebook:** `1.4.2_SAE_Circuits_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.4.2.11](https://delta-drills.vercel.app/?arena_exercise=1.4.2.11)


# [1.4.2] SAE Circuits (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/22_[1.4.2]_SAE_Circuits)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part42_sae_circuits/1.4.2_SAE_Circuits_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part42_sae_circuits/1.4.2_SAE_Circuits_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-13-2.png" width="350">
<br>

# Introduction

In these exercises, we explore **circuits with SAEs**: sets of SAE latents in different layers of a transformer which communicate with each other, and explain some particular model behaviour in an end-to-end way. We'll start by computing latent-to-latent, token-to-latent and latent-to-logit gradients, which give us a linear proxy for how latents in different layers are connected. We'll then move on to **transcoders**, a variant of SAEs which learn to reconstruct a model layer's computation rather than just its activations, and which offer significant advantages for circuit analysis.

We expect some degree of prerequisite knowledge in these exercises. Specifically, it will be very helpful if you understand:

- What **superposition** is, and what the **sparse autoencoder** architecture is (if you need a refresher on these topics, see exercises **1.5.4 Toy Models of SAEs & Superposition**)
- How to use the **SAELens** library to load and run SAEs alongside TransformerLens models (covered in the first section of exercises **1.3.3 Interpretability with SAEs**)

We've included a short section at the start to speedrun the most relevant background from exercise set 1.3.3 (mainly loading models & SAEs and running forward passes with them), so you don't need to have completed that exercise set in full before starting this one.

One note on terminology: we'll be mostly adopting the convention that **features** are characteristics of the underlying data distribution that our base models are trained on, and **SAE latents** (or just "latents") are the directions in the SAE. This is to avoid the overloading of the term "feature", and avoiding the implicit assumption that "SAE features" correspond to real features in the data. We'll relax this terminology when we're looking at SAE latents which very clearly correspond to specific interpretable features in the data.

## Reading Material

- [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html) arguably took the first major stride in mechanistic interpretability with SAEs: training them on a 1-layer model, and extracting a large number of interpretable features.
- [Scaling Monosemanticity: Extracting Interpretable Features from Claude 3 Sonnet](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html) shows how you can scale up the science of SAEs to larger models.
- [Transcoders Find Interpretable LLM Feature Circuits](https://arxiv.org/abs/2406.11944) (Dunefsky et al., 2024) introduces transcoders, a variant of SAEs that learn to map MLP inputs to MLP outputs rather than just reconstructing activations - making them well-suited for circuit analysis. This is the main technique used in Section 2 of this exercise set. Read at least the abstract and Section 2 ("Methods").
- [Circuit Tracing: Revealing Computational Graphs in Language Models](https://transformer-circuits.pub/2025/attribution-graphs/methods.html) (Anthropic, 2025) introduces the attribution graph framework for decomposing model computation into interpretable computational graphs using transcoders. Section 3 of this exercise set implements this from scratch. Read the "Methods" section for the technical details you'll implement.
- [On the Biology of a Large Language Model](https://transformer-circuits.pub/2025/attribution-graphs/biology.html) is the companion paper that applies attribution graphs to study diverse phenomena in Claude. Browse for motivation and to see what kinds of circuits attribution graphs can reveal.

## Content & Learning Objectives

### 1️⃣ Latent Gradients

SAEs are cool and interesting and we can steer on their latents to produce cool and interesting effects, but does this mean that we've truly unlocked the true units of computation used by our models, or have we just found an interesting clustering algorithm? The answer is that we don't really know yet! One strong piece of evidence for the former would be finding **circuits with SAEs**, in other words sets of latents in different layers of the transformer which communicate with each other, and explain some particular behaviour in an end-to-end way. In this section, we'll compute gradients between latents in different layers to build up a picture of how they communicate.

> ##### Learning Objectives
>
> - Learn how to compute **latent-to-latent gradients** between SAE latents in different layers of the transformer
> - Compute **token-to-latent gradients** to understand which input tokens drive particular latent activations
> - Compute **latent-to-logit gradients** to understand how latents affect the model's output
> - Use these gradient-based methods to find circuits in attention SAEs (e.g. induction circuits)

### 2️⃣ Transcoders

**Transcoders** are a variant of SAEs which learn to reconstruct a model layer's computation (e.g. a sparse mapping from MLP input to MLP output), rather than just reconstructing activations at a single point. They offer significant advantages for circuit analysis, since they decompose the function of an MLP layer into sparse, interpretable units. In this section, we'll load and work with transcoders, study their properties using techniques like de-embeddings, and go through a blind case study where we reverse-engineer a transcoder latent purely from weights-based analysis.

> ##### Learning Objectives
>
> - Understand transcoders, and how they differ from standard SAEs
> - Learn techniques for interpreting transcoder latents: **pullbacks**, **de-embeddings**, and **extended embeddings**
> - Work through a **blind case study**, interpreting a transcoder latent using only circuit-level analysis (no activation examples)

### 3️⃣ Attribution graphs

Attribution graphs extend the gradient-based methods from section 1️⃣ into a full framework for understanding end-to-end computation in transformers via transcoder latents. In this section, you'll implement the full attribution graph pipeline from scratch using Gemma 3-1B IT with GemmaScope 2 transcoders: linearising the model by freezing non-linearities, building the reading/writing vector abstraction for all node types, computing edge weights via batched backward passes, and pruning the graph using influence-based Neumann series propagation.

> ##### Learning Objectives
>
> - Understand the local replacement model: why freezing attention patterns, LayerNorm scales, and replacing MLPs with linear skip connections makes the residual stream linear
> - Understand the reading/writing vector abstraction: how token embeddings, transcoder latents, MLP errors, and logit directions all interact with the residual stream
> - Implement the core attribution algorithm: salient logit selection, graph node construction, and edge weight computation via gradient injection
> - Implement graph pruning via node and edge influence thresholding, using the Neumann series on the nilpotent adjacency matrix
> - Build interactive attribution graph visualisations using the same dashboard templates as Anthropic's published work

### 4️⃣ Exploring circuits & interventions

Now that you've built the attribution graph algorithm from scratch, you'll use the `circuit-tracer` library to explore real circuits and perform feature interventions. You'll study the Dallas/Austin two-hop factual recall circuit, test its causal structure via zero ablation, swap features between prompts, and generate text with feature interventions.

> ##### Learning Objectives
>
> - Load and inspect pre-computed attribution graphs and their supernodes
> - Perform zero ablation experiments to test causal claims made by the graph
> - Perform cross-prompt feature swapping to demonstrate compositional circuit structure
> - Use open-ended generation with feature interventions

## A note on memory usage

In these exercises, we'll be loading some pretty large models into memory (e.g. Gemma 2-2B and its SAEs, as well as a host of other models in later sections of the material). It's useful to have functions which can help profile memory usage for you, so that if you encounter OOM errors you can try and clear out unnecessary models. For example, we've found that with the right memory handling (i.e. deleting models and objects when you're not using them any more) it should be possible to run all the exercises in this material on a Colab Pro notebook, and all the exercises minus the handful involving Gemma on a free Colab notebook.

<details>
<summary>See this dropdown for some functions which you might find helpful, and how to use them.</summary>

First, we can run some code to inspect our current memory usage. Here's an example of running this code on a Colab Pro notebook.

```python
import part42_sae_circuits.utils as utils

# Profile memory usage, and delete gemma models if we've loaded them in
namespace = globals().copy() | locals()
utils.profile_pytorch_memory(namespace=namespace, filter_device="cuda:0")
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 35.88 GB
Total = 39.56 GB
Free = 3.68 GB
┌──────────────────────┬────────────────────────┬──────────┬─────────────┐
│ Name                 │ Object                 │ Device   │   Size (GB) │
├──────────────────────┼────────────────────────┼──────────┼─────────────┤
│ gemma_2_2b           │ HookedSAETransformer   │ cuda:0   │       11.94 │
│ gpt2                 │ HookedSAETransformer   │ cuda:0   │        0.61 │
│ gemma_2_2b_sae       │ SAE                    │ cuda:0   │        0.28 │
│ sae_resid_dirs       │ Tensor (4, 24576, 768) │ cuda:0   │        0.28 │
│ gpt2_sae             │ SAE                    │ cuda:0   │        0.14 │
│ logits               │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ logits_with_ablation │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ clean_logits         │ Tensor (4, 15, 50257)  │ cuda:0   │        0.01 │
│ _                    │ Tensor (16, 128, 768)  │ cuda:0   │        0.01 │
│ clean_sae_acts_post  │ Tensor (4, 15, 24576)  │ cuda:0   │        0.01 │
└──────────────────────┴────────────────────────┴──────────┴─────────────┘</pre>

From this, we see that we've allocated a lot of memory for the the Gemma model, so let's delete it. We'll also run some code to move any remaining objects on the GPU which are larger than 100MB to the CPU, and print the memory status again.

```python
del gemma_2_2b
del gemma_2_2b_sae

THRESHOLD = 0.1  # GB
for obj in gc.get_objects():
    try:
        if isinstance(obj, t.nn.Module) and utils.get_tensors_size(obj) / 1024**3 > THRESHOLD:
            if hasattr(obj, "cuda"):
                obj.cpu()
            if hasattr(obj, "reset"):
                obj.reset()
    except Exception:
        pass

# Move our gpt2 model & SAEs back to GPU (we'll need them for the exercises we're about to do)
gpt2.to(device)
gpt2_saes = {layer: sae.to(device) for layer, sae in gpt2_saes.items()}

utils.print_memory_status()
```

<pre style="font-family: Consolas; font-size: 14px">Allocated = 14.90 GB
Reserved = 39.56 GB
Free = 24.66</pre>

Mission success! We've managed to free up a lot of memory. Note that the code which moves all objects collected by the garbage collector to the CPU is often necessary to free up the memory. We can't just delete the objects directly because PyTorch can still sometimes keep references to them (i.e. their tensors) in memory. In fact, if you add code to the for loop above to print out `obj.shape` when `obj` is a tensor, you'll see that a lot of those tensors are actually Gemma model weights, even once you've deleted `gemma_2_2b`.

</details>

## Setup (don't read, just run)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install "openai==1.56.1" einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" openai tabulate umap-learn hdbscan eindex-callum git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python git+https://github.com/callummcdougall/sae_vis.git@callum/v3 transformer_lens==2.17.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import gc
import os
import sys
from collections import Counter, namedtuple
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Callable, TypeAlias

import einops
import numpy as np
import plotly.express as px
import torch as t
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import IFrame, display
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from sae_lens import SAE, ActivationsStore, HookedSAETransformer
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache, HookedTransformer
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt, to_numpy

dtype = t.float32  # t.bfloat16
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
device = str(device)  # SAELens expects device as string; this is easier!


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])


# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part42_sae_circuits"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part42_sae_circuits.tests as tests
import part42_sae_circuits.utils as utils

MAIN = __name__ == "__main__"

In [ ]:
# For displaying sae-vis inline
if IN_COLAB:
    import http.server
    import socketserver
    import threading

    from google.colab import output

    PORT = 8000

    def display_vis_inline(filename: Path, height: int = 850):
        """
        Displays the HTML files in Colab. Uses global `PORT` variable defined in prev cell, so that each
        vis has a unique port without having to define a port within the function.
        """
        global PORT

        def serve(directory):
            os.chdir(directory)
            handler = http.server.SimpleHTTPRequestHandler
            with socketserver.TCPServer(("", PORT), handler) as httpd:
                print(f"Serving files from {directory} on port {PORT}")
                httpd.serve_forever()

        thread = threading.Thread(target=serve, args=("/content",))
        thread.start()

        filename = str(filename).split("/content")[-1]

        output.serve_kernel_port_as_iframe(PORT, path=filename, height=height, cache_in_notebook=True)

        PORT += 1

## Speedrunning some relevant background

This section covers the essential SAELens concepts you'll need for the rest of these exercises. **If you've already completed exercise set 1.3.3 Interpretability with SAEs, you can skip this section and go straight to section 1️⃣.**

### Loading SAEs with `SAE.from_pretrained`

[SAELens](https://github.com/jbloomAus/SAELens) is a library designed to help researchers train and analyse sparse autoencoders. You can think of it as the equivalent of TransformerLens for sparse autoencoders (and it also integrates very well with TransformerLens models, which we'll see shortly).

To load an SAE, use `SAE.from_pretrained`. This function returns an SAE object directly:

```python
gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)
```

You can view the available SAE releases in SAELens with `get_pretrained_saes_directory()`. Each release contains multiple SAEs (e.g. trained on different layers of the same base model).

Base models are loaded using the `HookedSAETransformer` class, which is adapted from the TransformerLens `HookedTransformer` class:

```python
gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device)
```

### Running SAEs and caching activations

You can add SAEs to a TransformerLens model when doing forward passes in much the same way you add hook functions. There are several different methods for this:

- **`model.run_with_saes(tokens, saes=[list_of_saes])`** works like `model.run_with_hooks`, doing a single forward pass with SAEs attached (then resetting them).
- **`logits, cache = model.run_with_cache_with_saes(tokens, saes=[sae])`** works like `model.run_with_cache`, caching all intermediate activations including SAE activations.
- **`with model.saes(saes=[sae]):`** is a context manager which temporarily attaches SAEs.
- **`model.add_sae(sae)` / `model.reset_saes()`** manually adds and removes SAEs.

To access SAE activations from the cache, the hook names are the concatenation of the HookedTransformer `hook_name` and the SAE hook name, joined by a period. The most important ones are:

```python
# Post-activation latent values (shape [batch, seq, d_sae]) - this is the one you'll use most
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"]

# Pre-activation latent values (before the activation function)
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_acts_pre"]

# The SAE's reconstruction of the original activations
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_recons"]

# The final SAE output (either reconstruction, or reconstruction + error term)
cache[f"{sae.cfg.metadata.hook_name}.hook_sae_output"]
```

Here's a full example, extracting the top activating latents at the final token of a prompt:

```python
_, cache = gpt2.run_with_cache_with_saes(
    prompt,
    saes=[gpt2_sae],
    stop_at_layer=_get_hook_layer(gpt2_sae) + 1,  # no need to compute past the SAE layer
)
sae_acts_post = cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, :]
```

### The `use_error_term` parameter

The parameter `sae.use_error_term` determines whether we actually substitute the model's activations with SAE reconstructions during the forward pass:

- **`use_error_term=False`** (default): the SAE's output replaces the transformer's activations. This means downstream computations use the SAE reconstruction rather than the original activations.
- **`use_error_term=True`**: the SAE computes all its internal states (latent activations etc.) in the same way, but the transformer's activations are left intact. This is useful when you want to cache SAE activations without actually intervening on the model.

```python
# Cache SAE activations WITHOUT intervening on the model
gpt2_sae.use_error_term = True
logits, cache = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])
# logits are identical to running the base model without SAEs, but we still get SAE activations in the cache

# Cache SAE activations AND replace model activations with SAE reconstructions
gpt2_sae.use_error_term = False
logits_recon, cache_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])
# logits_recon will differ from the base model's logits due to SAE reconstruction error
```

This parameter matters a lot for the gradient exercises in this set: we'll typically use `use_error_term=True` to get the "true" latent activations from a clean forward pass, then `use_error_term=False` when computing Jacobians (because we want our Jacobian function to actually pass through the SAE decoder and encoder).

### Using `ActivationsStore`

The `ActivationsStore` class is a convenient alternative to loading a bunch of data yourself. It streams in data from a given dataset; in the case of the `from_sae` classmethod that dataset will be given by your SAE's config (which is also the same as the SAE's original training dataset):

```python
gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_sae,
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=str(device),
)

# Get a batch of tokens
tokens = gpt2_act_store.get_batch_tokens()
assert tokens.shape == (gpt2_act_store.store_batch_size_prompts, gpt2_act_store.context_size)
```

### Neuronpedia & `display_dashboard`

[Neuronpedia](https://neuronpedia.org) is an open platform for interpretability research. It hosts **SAE dashboards** which help you quickly understand what a particular SAE latent represents, including components like max activating examples, top logits, activation density plots, and LLM-generated explanations.

We can display these dashboards inline using the following helper function, which we'll use in several places throughout these exercises:

In [ ]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=800,
    height=600,
) -> None:
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))

# 1️⃣ Latent Gradients

> ##### Learning Objectives
>
> - Learn how to compute **latent-to-latent gradients** between SAE latents in different layers of the transformer
> - Compute **token-to-latent gradients** to understand which input tokens drive particular latent activations
> - Compute **latent-to-logit gradients** to understand how latents affect the model's output
> - Use these gradient-based methods to find circuits in attention SAEs (e.g. induction circuits)

## Introduction

Previous SAE work (in material 1.3.3) has focused on understanding individual latents. Here, we address a highly important topic - what about **circuits of SAE latents**? Circuit analysis has already been somewhat successful in language model interpretability (e.g. see Anthropic's work on induction circuits, or the Indirect Object Identification paper), but many attempts to push circuit analysis further have hit speedbumps: most connections in the model are not sparse, and it's very hard to disentangle all the messy cross-talk between different components and residual stream subspaces. Circuits offer a better path forward, since we should expect that not only are individual latents generally sparse, they are also **sparsely connected** - any given latent should probably only have a downstream effect on a small number of other latents.

Indeed, if this does end up being true, it's a strong piece of evidence that latents found by SAEs *are* the **fundamental units of computation** used by the model, as opposed to just being an interesting clustering algorithm. Of course we do already have some evidence for this (e.g. the effectiveness of latent steering, and the fact that latents have already revealed important information about models which isn't clear when just looking at the basic components), but finding clear latent circuits would be a much stronger piece of evidence.

## Latent Gradients

We'll start with an exercise that illustrates the kind of sparsity you can expect in latent connections, as well as many of the ways latent circuit analysis can be challenging. We'll be implementing the `latent_to_latent_gradients` function, which returns the gradients between all active pairs of latents belonging to SAEs in two different layers (we'll be using two SAEs from our `gpt2-small-res-jb` release). These exercises will be split up into a few different steps, since computing these gradients is deceptively complex.

What exactly are latent gradients? Well, for any given input, and any 2 latents in different layers, we can compute the derivative of the second latent's activation with respect to the first latent. This takes the form of a matrix of partial derivatives, i.e. $J_{ij} = \frac{\partial f_i}{\partial x_j}$, and can serve as a linear proxy for how latents in an early layer contribute to latents in a later layer. The pseudocode for computing this is:

```python
# Computed with no gradients, and not patching in SAE reconstructions...
layer_1_latents, layer_2_latents = model.run_with_cache_with_saes(...)

def latent_acts_to_later_latent_acts(layer_1_latents):
    layer_1_resid_acts_recon = SAE_1_decoder(layer_1_latents)
    layer_2_resid_acts_recon = model.blocks[layer_1: layer_2].forward(layer_1_resid_acts_recon)
    layer_2_latents_recon = SAE_2_encoder(layer_2_resid_acts_recon)
    return layer_2_latents_recon

latent_latent_gradients = t.func.jacrev(latent_acts_to_later_latent_acts)(layer_1_latents)
```

where `jacrev` is shorthand for "Jacobian reverse-mode differentiation" - it's a PyTorch function that takes in a tensor -> tensor function `f(x) = y` and returns the Jacobian function, i.e. `g` s.t. `g[i, j] = d(f[x]_i) / d(x_j)`.

If we wanted to get a sense of how latents communicate with each other across our distribution of data, then we might average these results over a large set of prompts. However for now, we're going to stick with a relatively small set of prompts to avoid running into memory issues, and so we can visualise the results more easily.

First, let's load in our model & SAEs, if you haven't already:

In [ ]:
gpt2 = HookedSAETransformer.from_pretrained("gpt2-small", device=device, dtype=dtype)

gpt2_saes = {
    layer: SAE.from_pretrained(
        release="gpt2-small-res-jb",
        sae_id=f"blocks.{layer}.hook_resid_pre",
        device=device,
        dtype=dtype,
    )
    for layer in tqdm(range(gpt2.cfg.n_layers))
}

Now, we can start the exercises!

Note - the subsequent 3 exercises are all somewhat involved, and things like the use of Jacobian can be quite fiddly. For that reason, there's a good case to be made for just reading through the solutions and understanding what the code is doing, rather than trying to do it yourself. One option would be to look at the solutions for these 3 exercises and understand how latent-to-latent gradients work, but then try and implement the `token_to_latent_gradients` function (after the next 3 exercises) yourself.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.4.2.11"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part42_sae_circuits.solutions import SparseTensor, latent_acts_to_later_latent_acts, latent_to_latent_gradients, tokens_to_latent_acts, latent_acts_to_logits, latent_acts_to_later_latent_acts_attn, show_top_logits, create_extended_embedding, TranscoderReplacementHooks, compute_salient_logits, build_embedding_nodes


### Exercise (2/3) - implement `build_intermediate_nodes`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

Next, we build the intermediate nodes: for each layer and each position (from `start_posn` onwards), we select the top `top_k` transcoder latents by activation and create a LATENT node for each one. We also create one MLP_ERROR node per position per layer to capture the reconstruction error.

We've given you the outer loop structure (iterating over layers/positions and extracting the active feature indices). You need to fill in the inner loop body that creates the actual nodes and their vectors:

- For each active feature: create a LATENT node with writing vector `activation * W_dec[feature_idx]` (the activation-scaled decoder direction) and reading vector `W_enc.T[feature_idx]` (the encoder direction).
- After processing all features at a given position: create one MLP_ERROR node with writing vector `mlp_output[pos] - tc_output[pos]` and reading vector zeros.

<details><summary>Hint - computing the MLP error</summary>

You can get the MLP output from the cache using `tc.cfg.metadata.hook_name_out` (e.g. `"blocks.0.hook_mlp_out"`). Then the error is `mlp_output[pos] - tc_output[pos]`.

</details>

In [ ]:
def build_intermediate_nodes(
    transcoders: dict[int, "Transcoder"],
    cache: ActivationCache,
    tc_hooks: TranscoderReplacementHooks,
    tokens: Int[Tensor, "1 seq"],
    d_model: int,
    start_posn: int = 4,
    top_k: int = 5,
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor], dict]:
    """
    Build intermediate (LATENT + MLP_ERROR) nodes for all layers.

    Args:
        transcoders: Dict mapping layer -> transcoder.
        cache: Activation cache from forward pass.
        tc_hooks: TranscoderReplacementHooks with computed activations.
        tokens: Input token IDs, shape (1, seq).
        d_model: Model hidden dimension.
        start_posn: First position to include (masks chat formatting tokens).
        top_k: Number of top-activating features to include per position per layer.

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs, node_range_dict) where node_range_dict
        maps each layer index to (start_idx, end_idx) within the returned lists.
    """
    seq_len = tokens.shape[1]
    n_layers = len(transcoders)
    device = cache["blocks.0.hook_resid_pre"].device

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []
    node_range_dict = {}

    for layer in range(n_layers):
        tc = transcoders[layer]
        layer_start = len(nodes)

        tc_acts = tc_hooks.transcoder_acts[layer][0]  # (seq, d_enc)
        tc_output = tc_hooks.transcoder_output[layer][0]  # (seq, d_model)

        # Get MLP output from cache for error computation
        mlp_output = cache[tc.cfg.metadata.hook_name_out][0]  # (seq, d_model)

        W_dec = tc.W_dec.detach()  # (d_enc, d_model)
        W_enc_T = tc.W_enc.detach().T  # (d_enc, d_model)

        for pos in range(start_posn, seq_len):
            acts = tc_acts[pos]  # (d_enc,)
            # Select top-k features by activation
            k = min(top_k, (acts > 0).sum().item())  # don't include zero-activation features
            if k > 0:
                _, active_indices = acts.topk(k)
            else:
                active_indices = t.where(acts > 0)[0]  # fallback: empty if no active features

            # TODO: For each active feature, create a LATENT node with the correct writing
            # vector (activation * W_dec[feature]) and reading vector (W_enc.T[feature]).
            # After all features at this position, create one MLP_ERROR node with writing
            # vector = mlp_output[pos] - tc_output[pos] and reading vector = zeros.
            pass

        node_range_dict[layer] = (layer_start, len(nodes))

    return nodes, writing_vecs, reading_vecs, node_range_dict


inter_nodes, inter_writing, inter_reading, inter_range_dict = build_intermediate_nodes(
    transcoders=transcoders,
    cache=cache,
    tc_hooks=tc_hooks,
    tokens=tokens,
    d_model=gemma.cfg.d_model,
    start_posn=START_POSN,
)

n_layers = len(transcoders)
seq_len = tokens.shape[1]
active_positions = seq_len - START_POSN

# All LATENT nodes should have positive activations
latent_nodes = [n for n in inter_nodes if n.node_type == NodeType.LATENT]
assert len(latent_nodes) > 0, "Expected at least some LATENT nodes"
assert all(n.activation > 0 for n in latent_nodes), "All LATENT nodes should have positive activations"

# MLP_ERROR nodes: one per position per layer
error_nodes = [n for n in inter_nodes if n.node_type == NodeType.MLP_ERROR]
assert len(error_nodes) == active_positions * n_layers, (
    f"Expected {active_positions * n_layers} MLP_ERROR nodes, got {len(error_nodes)}"
)

# Writing vectors for LATENT nodes should be non-zero
latent_indices = [i for i, n in enumerate(inter_nodes) if n.node_type == NodeType.LATENT]
for idx in latent_indices[:5]:  # check first 5
    assert inter_writing[idx].abs().sum() > 0, "LATENT writing vectors should be non-zero"

# Reading vectors for MLP_ERROR nodes should be zeros
error_indices = [i for i, n in enumerate(inter_nodes) if n.node_type == NodeType.MLP_ERROR]
for idx in error_indices[:5]:  # check first 5
    t.testing.assert_close(inter_reading[idx], t.zeros_like(inter_reading[idx]))

# node_range_dict should have entries for each layer
assert set(inter_range_dict.keys()) == set(range(n_layers)), (
    f"node_range_dict should have entries for layers 0..{n_layers - 1}, got keys {set(inter_range_dict.keys())}"
)

print("All build_intermediate_nodes tests passed!")

<details><summary>Solution</summary>

```python
def build_intermediate_nodes(
    transcoders: dict[int, "Transcoder"],
    cache: ActivationCache,
    tc_hooks: TranscoderReplacementHooks,
    tokens: Int[Tensor, "1 seq"],
    d_model: int,
    start_posn: int = 4,
    top_k: int = 5,
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor], dict]:
    """
    Build intermediate (LATENT + MLP_ERROR) nodes for all layers.

    Args:
        transcoders: Dict mapping layer -> transcoder.
        cache: Activation cache from forward pass.
        tc_hooks: TranscoderReplacementHooks with computed activations.
        tokens: Input token IDs, shape (1, seq).
        d_model: Model hidden dimension.
        start_posn: First position to include (masks chat formatting tokens).
        top_k: Number of top-activating features to include per position per layer.

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs, node_range_dict) where node_range_dict
        maps each layer index to (start_idx, end_idx) within the returned lists.
    """
    seq_len = tokens.shape[1]
    n_layers = len(transcoders)
    device = cache["blocks.0.hook_resid_pre"].device

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []
    node_range_dict = {}

    for layer in range(n_layers):
        tc = transcoders[layer]
        layer_start = len(nodes)

        tc_acts = tc_hooks.transcoder_acts[layer][0]  # (seq, d_enc)
        tc_output = tc_hooks.transcoder_output[layer][0]  # (seq, d_model)

        # Get MLP output from cache for error computation
        mlp_output = cache[tc.cfg.metadata.hook_name_out][0]  # (seq, d_model)

        W_dec = tc.W_dec.detach()  # (d_enc, d_model)
        W_enc_T = tc.W_enc.detach().T  # (d_enc, d_model)

        for pos in range(start_posn, seq_len):
            acts = tc_acts[pos]  # (d_enc,)
            # Select top-k features by activation
            k = min(top_k, (acts > 0).sum().item())  # don't include zero-activation features
            if k > 0:
                _, active_indices = acts.topk(k)
            else:
                active_indices = t.where(acts > 0)[0]  # fallback: empty if no active features

            for feat_idx in active_indices:
                feat_act = acts[feat_idx].item()
                nodes.append(
                    NodeInfo(
                        node_type=NodeType.LATENT,
                        layer=layer,
                        ctx_idx=pos,
                        feature=feat_idx.item(),
                        activation=feat_act,
                    )
                )
                # Writing vector: activation * decoder direction
                writing_vecs.append(feat_act * W_dec[feat_idx])
                # Reading vector: encoder direction
                reading_vecs.append(W_enc_T[feat_idx])

            # MLP error node for this position
            mlp_error = mlp_output[pos] - tc_output[pos]
            nodes.append(
                NodeInfo(
                    node_type=NodeType.MLP_ERROR,
                    layer=layer,
                    ctx_idx=pos,
                    feature=0,
                )
            )
            writing_vecs.append(mlp_error.detach())
            reading_vecs.append(t.zeros(d_model, device=device))

        node_range_dict[layer] = (layer_start, len(nodes))

    return nodes, writing_vecs, reading_vecs, node_range_dict
```
</details>

Finally, we build the output (logit) nodes. There is one node per entry in `top_token_info`, positioned at the last sequence position. These nodes have zero writing vectors (they don't write to the residual stream) and their reading vectors are the demeaned unembedding columns from `reading_vecs_logit`. We've given you this function since it follows the same pattern as `build_embedding_nodes`.

In [ ]:
def build_logit_nodes(
    reading_vecs_logit: Float[Tensor, "n_output d_model"],
    top_token_info: list[tuple[str, float]],
    seq_len: int,
    d_model: int,
) -> tuple[list[NodeInfo], list[Tensor], list[Tensor]]:
    """
    Build logit (output) nodes with their writing and reading vectors.

    Args:
        reading_vecs_logit: Demeaned W_U columns for output nodes, shape (n_output, d_model).
        top_token_info: List of (token_string, probability) for output nodes.
        seq_len: Total sequence length (logit nodes are placed at seq_len - 1).
        d_model: Model hidden dimension.

    Returns:
        Tuple of (nodes, writing_vecs, reading_vecs) where each list has length len(top_token_info).
        Writing vecs are zeros, reading vecs are the logit reading vectors.
    """
    device = reading_vecs_logit.device

    nodes: list[NodeInfo] = []
    writing_vecs: list[Tensor] = []
    reading_vecs: list[Tensor] = []

    for i, (tok_str, prob) in enumerate(top_token_info):
        nodes.append(
            NodeInfo(
                node_type=NodeType.LOGIT,
                layer="L",
                ctx_idx=seq_len - 1,
                feature=i,
                token_prob=prob,
                str_token=tok_str,
            )
        )
        writing_vecs.append(t.zeros(d_model, device=device))
        reading_vecs.append(reading_vecs_logit[i])

    return nodes, writing_vecs, reading_vecs

Now that all three helper functions are done, we combine them into a single `build_graph_nodes` function. This wrapper is provided for you and just calls the three sub-functions, assembles the results, and returns a `GraphNodes` object.

In [ ]:
def build_graph_nodes(
    model: HookedSAETransformer,
    transcoders: dict[int, "Transcoder"],
    cache: ActivationCache,
    tc_hooks: TranscoderReplacementHooks,
    reading_vecs_logit: Float[Tensor, "n_output d_model"],
    top_token_info: list[tuple[str, float]],
    tokens: Int[Tensor, "1 seq"],
    start_posn: int = 4,
    top_k: int = 5,
) -> GraphNodes:
    """
    Build all graph nodes with their reading and writing vectors by calling the three sub-functions
    and assembling the results.

    Args:
        model: The transformer model.
        transcoders: Dict mapping layer -> transcoder.
        cache: Activation cache from forward pass.
        tc_hooks: TranscoderReplacementHooks with computed activations.
        reading_vecs_logit: Demeaned W_U columns for output nodes.
        top_token_info: List of (token_string, probability) for output nodes.
        tokens: Input token IDs, shape (1, seq).
        start_posn: First position to include (masks chat formatting tokens).
        top_k: Number of top-activating features to include per position per layer.

    Returns:
        GraphNodes containing all nodes, their reading/writing vectors, and index ranges.
    """
    seq_len = tokens.shape[1]
    n_layers = len(transcoders)
    d_model = model.cfg.d_model

    all_nodes: list[NodeInfo] = []
    all_writing: list[Tensor] = []
    all_reading: list[Tensor] = []
    node_range_dict = {}

    # 1. Embedding nodes
    embed_nodes, embed_writing, embed_reading = build_embedding_nodes(model, cache, tokens)
    all_nodes.extend(embed_nodes)
    all_writing.extend(embed_writing)
    all_reading.extend(embed_reading)
    node_range_dict["E"] = (0, len(all_nodes))

    # 2. Intermediate nodes (features + MLP error per layer)
    inter_nodes, inter_writing, inter_reading, inter_ranges = build_intermediate_nodes(
        transcoders,
        cache,
        tc_hooks,
        tokens,
        d_model,
        start_posn,
        top_k,
    )
    offset = len(all_nodes)
    all_nodes.extend(inter_nodes)
    all_writing.extend(inter_writing)
    all_reading.extend(inter_reading)
    for layer_key, (s, e) in inter_ranges.items():
        node_range_dict[layer_key] = (s + offset, e + offset)

    # 3. Logit nodes
    logit_start = len(all_nodes)
    logit_nodes, logit_writing, logit_reading = build_logit_nodes(
        reading_vecs_logit,
        top_token_info,
        seq_len,
        d_model,
    )
    all_nodes.extend(logit_nodes)
    all_writing.extend(logit_writing)
    all_reading.extend(logit_reading)
    node_range_dict["L"] = (logit_start, len(all_nodes))

    return GraphNodes(
        nodes=all_nodes,
        writing_vecs=t.stack(all_writing),
        reading_vecs=t.stack(all_reading),
        node_range_dict=node_range_dict,
        seq_len=seq_len,
        n_layers=n_layers,
    )

In [ ]:
graph = build_graph_nodes(
    model=gemma,
    transcoders=transcoders,
    cache=cache,
    tc_hooks=tc_hooks,
    reading_vecs_logit=reading_vecs,
    top_token_info=top_token_info,
    tokens=tokens,
    start_posn=START_POSN,
)

# Print some stats
n_embeds = sum(1 for n in graph.nodes if n.node_type == NodeType.EMBEDDING)
n_latents = sum(1 for n in graph.nodes if n.node_type == NodeType.LATENT)
n_errors = sum(1 for n in graph.nodes if n.node_type == NodeType.MLP_ERROR)
n_logits = sum(1 for n in graph.nodes if n.node_type == NodeType.LOGIT)
print(f"Graph has {len(graph.nodes)} total nodes:")
print(f"  {n_embeds} embedding nodes")
print(f"  {n_latents} latent nodes")
print(f"  {n_errors} MLP error nodes")
print(f"  {n_logits} logit output nodes")
print(f"Writing vectors shape: {graph.writing_vecs.shape}")
print(f"Reading vectors shape: {graph.reading_vecs.shape}")

### Attribution setup

Before computing the adjacency matrix, we need a helper that sets up the backward pass properly. The idea is: (1) run a forward pass through the frozen model (with `FreezeHooks` and `TranscoderReplacementHooks` active), (2) at the target node's position and layer, compute the dot product of the residual stream with the target's reading vector, (3) backpropagate this scalar through the frozen model, and (4) read off the gradients at each source node's position and contract them with the source's writing vector.

We'll give you this setup function, which handles steps 1-3 for a batch of target nodes. Read through it carefully.

**Hook design note (Option A).** Rather than nesting `with freeze:` inside the function and separately installing/removing gradient-capture hooks, `setup_attribution` uses a single `model.hooks()` call that combines everything:

```python
with model.hooks(fwd_hooks=freeze.fwd_hooks + capture_fwd_hooks, bwd_hooks=capture_bwd_hooks):
    model(tokens).backward()
```

- `freeze.fwd_hooks` - returns the freeze hooks as `(name, fn)` pairs (TL-native format)
- `capture_fwd_hooks` - store each residual-stream activation tensor so we can build objectives
- `capture_bwd_hooks` - capture gradients **directly during the backward pass**, eliminating the need for `tensor.retain_grad()` + `tensor.grad` bookkeeping

`TranscoderReplacementHooks` uses **permanent** hooks (registered with `is_permanent=True`) so they
remain active throughout the `model.hooks()` context and are only removed by `tc_hooks.remove()`.

In [ ]:
def setup_attribution(
    model: HookedSAETransformer,
    tokens: Tensor,
    freeze: FreezeHooks,
    target_reading_vecs: Float[Tensor, "batch d_model"],
    target_positions: Int[Tensor, " batch"],
    target_layers: list[int | str],
) -> dict[str, Tensor]:
    """
    Run forward pass through frozen model, inject reading vectors as gradient seeds at
    target positions/layers, and return gradients at all residual stream positions.

    This function handles the "backward pass" part of attribution: for each target node,
    it computes d(objective)/d(resid) at every layer, where objective = dot(resid[target_pos], reading_vec).

    Uses TransformerLens-native hooks via a single model.hooks() context (Option A):
      - fwd_hooks = freeze.fwd_hooks + capture_fwd_hooks
          freeze.fwd_hooks  : replace attention patterns/LN scales with frozen values
          capture_fwd_hooks : store each residual-stream tensor for objective computation
      - bwd_hooks = capture_bwd_hooks
          capture_bwd_hooks : receive gradients directly during backward, no retain_grad() needed

    TranscoderReplacementHooks are permanent and stay active inside this context.

    Args:
        model: The transformer model.
        tokens: Input tokens, shape (1, seq).
        freeze: FreezeHooks with cached frozen values (provides fwd_hooks property).
        target_reading_vecs: Reading vectors for target nodes, shape (batch, d_model).
        target_positions: Sequence positions of target nodes, shape (batch,).
        target_layers: Layer identifiers for target nodes (int for intermediate, "L" for logits).

    Returns:
        grads: Dict mapping hook names -> gradient tensors at each residual stream position.
    """
    # Detach reading vectors - attribution only needs d(objective)/d(resid),
    # not d(objective)/d(W_U). Without this, reading_vecs built from W_U (a
    # parameter with requires_grad=True) carry a grad_fn that gets freed after
    # batch 1's backward(), causing "backward through graph a second time" on
    # subsequent batches.
    target_reading_vecs = target_reading_vecs.detach()

    batch_size = target_reading_vecs.shape[0]

    # Tensors captured during the forward pass (for computing objectives)
    captured_tensors: dict[str, Tensor] = {}
    # Gradients captured directly during the backward pass (the actual output)
    captured_grads: dict[str, Tensor] = {}

    # Names of residual-stream tensors to capture
    capture_names = (
        [f"blocks.{l}.hook_resid_post" for l in range(model.cfg.n_layers)]
        + [f"blocks.{l}.hook_resid_mid" for l in range(model.cfg.n_layers)]
        + ["blocks.0.hook_resid_pre"]
    )

    def make_fwd_capture(name: str) -> Callable:
        """Forward hook: store the activation tensor so we can build objectives from it."""

        def hook_fn(tensor: Tensor, hook: HookPoint) -> None:
            captured_tensors[name] = tensor

        return hook_fn

    def make_bwd_capture(name: str) -> Callable:
        """Backward hook: receive the gradient directly during backward, no retain_grad() needed."""

        def hook_fn(grad: Tensor, hook: HookPoint) -> None:
            captured_grads[name] = grad.detach()

        return hook_fn

    capture_fwd_hooks = [(name, make_fwd_capture(name)) for name in capture_names]
    capture_bwd_hooks = [(name, make_bwd_capture(name)) for name in capture_names]

    # Single model.hooks() context combining freeze hooks + capture hooks (Option A).
    # TranscoderReplacementHooks are permanent and stay active throughout.
    # At context exit, only non-permanent hooks (freeze + capture) are removed.
    with model.hooks(
        fwd_hooks=freeze.fwd_hooks + capture_fwd_hooks,
        bwd_hooks=capture_bwd_hooks,
    ):
        # Forward pass (objectives are computed from captured residual tensors)
        model(tokens.expand(batch_size, -1))

        # Compute objectives for each target node
        # Use torch.stack instead of in-place assignment to a leaf tensor, to
        # avoid autograd issues with in-place ops on zero-initialized tensors.
        objective_terms = []
        for i in range(batch_size):
            pos = target_positions[i]
            layer = target_layers[i]
            if layer == "L":
                # For logit nodes, use the final residual stream (pre-unembedding)
                resid = captured_tensors[f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"][i, pos]
            else:
                # For intermediate nodes, use resid_mid at this layer
                # (after attention, before MLP - this is what the feature reads from)
                resid = captured_tensors[f"blocks.{layer}.hook_resid_mid"][i, pos]
            objective_terms.append((resid * target_reading_vecs[i]).sum())

        # Backward pass - fires bwd hooks, populating captured_grads
        total_objective = t.stack(objective_terms).sum()
        total_objective.backward()

    return captured_grads

We'll implement the attribution algorithm in two steps. First we prepare the backward pass batches (grouping target nodes by layer and batching their reading vectors), then we run the actual backward passes and contract the gradients with source writing vectors.

The adjacency matrix `A[target, source]` stores the edge weight from source to target. Since our nodes are ordered by layer, this matrix is **strictly lower triangular** (a target can only receive from earlier layers).

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
